In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from shapely.affinity import translate  # Para mover geometrías
import re
import folium

In [ ]:
import os
# Cambiar el directorio de trabajo
os.chdir("C:/PARA EL EJEMPLO DE MOVILIDAD")

In [ ]:
#######################################
# LEER EL FICHERO DE MOVILIDAD        #
#######################################
# Leer shapefile de las áreas de movilidad
areas_movilidad = gpd.read_file("shapefiles_areas_movilidad/celdas_marzo_2020.shp", encoding="utf-8")

In [ ]:
print(areas_movilidad)

In [ ]:
#######################################
# PARÁMETROS                          #
#######################################

ruta_base = r"C:\movilidad_cotidiana_enero_noviembre_2021"
ruta_resultados = os.path.join(ruta_base, "Resultados")
ruta_codigos = os.path.join(ruta_base, "CodigoAreaMovilidad.csv")

# Día elegido: 
dia_elegido = "0404"

# Método: "Leiden" o "Louvain"
metodo = "Leiden"

# Alpha elegida: de 1.00 a 0.00 en pasos de 0.01
alpha_objetivo = 0.95

In [ ]:
#######################################
#  PROCESO                           #
#######################################

In [ ]:
# COMPROBACIONES

if not os.path.isdir(ruta_resultados):
    raise FileNotFoundError(f"No existe la carpeta de resultados: {ruta_resultados}")

if not os.path.isfile(ruta_codigos):
    raise FileNotFoundError(f"No existe el archivo: {ruta_codigos}")

if metodo not in ["Leiden", "Louvain"]:
    raise ValueError("El parámetro 'metodo' debe ser 'Leiden' o 'Louvain'")

if not (0 <= alpha_objetivo <= 1):
    raise ValueError("alpha_objetivo debe estar entre 0 y 1")

In [ ]:
# LOCALIZAR EXCEL AUTOMÁTICAMENTE
patron = re.compile(
    rf"^{metodo}_AD_.*{dia_elegido}\.xlsx$",
    flags=re.IGNORECASE
)

archivos_candidatos = [
    f for f in os.listdir(ruta_resultados)
    if patron.match(f)
]

if len(archivos_candidatos) == 0:
    raise FileNotFoundError(
        f"No se encontró ningún archivo que cumpla el patrón para método={metodo} y día={dia_elegido}"
    )

if len(archivos_candidatos) > 1:
    print("Se encontraron varios archivos candidatos:")
    for f in archivos_candidatos:
        print(" -", f)
    print("\nSe usará el primero.")

archivo_excel = os.path.join(ruta_resultados, archivos_candidatos[0])

print("Archivo seleccionado:")
print(archivo_excel)


In [ ]:
# LEER HOJA 'comunidades'

df_com = pd.read_excel(archivo_excel, sheet_name="comunidades")

if "alpha" not in df_com.columns:
    raise ValueError("La hoja 'comunidades' no contiene una columna llamada 'alpha'")


In [ ]:
# LOCALIZAR FILA DEL ALPHA

# Convertimos alpha a numérico por seguridad
df_com["alpha"] = pd.to_numeric(df_com["alpha"], errors="coerce")
# Seleccionamos la fila del alpha elegido
fila_alpha = df_com[df_com["alpha"] == alpha_objetivo]
fila_alpha = fila_alpha.iloc[0]

In [ ]:
# EXTRAER VECTOR DE CLUSTERS

# Desde la columna 3 en adelante (posición 2 en Python),
cluster = fila_alpha.iloc[2:].reset_index(drop=True)

In [ ]:
# LEER CODIGOAREA MOVILIDAD

equiv = pd.read_csv(ruta_codigos, sep=";", encoding="utf-8")

In [ ]:
cluster_sel = pd.DataFrame({
    "ID_GRUPO": equiv.loc[:, "CodigoArea"].values,
    "cluster": cluster.iloc[:].values
})

print(cluster_sel)

In [ ]:
#######################################
#  UNIR COMUNIDADES Y SHAPEFILE       #
#######################################

In [ ]:
# Hacer un merge con las áreas de movilidad
areas_cluster = areas_movilidad.merge(cluster_sel, on="ID_GRUPO", how="inner")
print(areas_cluster)

In [ ]:
colores = [
    "#B80000", "#8B0000", "#DB7400", "#EBBA2E", "saddlebrown",
    "#F8FBA5", "darkviolet", "#CCFF66", "#A7FF00", "#669900",  
    "#99CC00", "#194D00", "#002C1A", "#0EFF00", "#00FF7A", 
    "lightcoral", "#00FFFF", "lightgray", "dimgray", "#0080FF", 
    "#004DFF", "#0000FF",  "#000080", "#000033", "#002233",
    "#004488", "#6666FF", "#9999FF", "#CCCCFF", "#CCFF99",
    "#CCFFCC"
]

In [ ]:

#######################################
#  ASIGNAR COLORES ALEATORIOS         #
#######################################

rng = np.random.default_rng()

# Obtener clusters únicos
clusters_unicos = sorted(pd.Series(areas_cluster["cluster"]).dropna().unique())

# Función para generar colores hex aleatorios
def color_hex_aleatorio(rng):
    return "#{:02X}{:02X}{:02X}".format(
        int(rng.integers(0, 256)),
        int(rng.integers(0, 256)),
        int(rng.integers(0, 256))
    )

# Crear tantos colores como comunidades haya
colores_random = [color_hex_aleatorio(rng) for _ in range(len(clusters_unicos))]

# Diccionario cluster -> color
mapa_colores = dict(zip(clusters_unicos, colores_random))

# Añadir color al GeoDataFrame
areas_cluster["color"] = areas_cluster["cluster"].map(mapa_colores)

# Vista rápida de la asignación
tabla_colores = pd.DataFrame({
    "cluster": clusters_unicos,
    "color": [mapa_colores[c] for c in clusters_unicos]
})

tabla_colores.head()

In [ ]:

#######################################
#  MAPA INTERACTIVO DE ESPAÑA         #
#######################################

# Instalar folium si hiciera falta
# !pip install folium



# Copia para el mapa y conversión de CRS a WGS84 (lat/lon)
areas_cluster_map = areas_cluster.copy()

if areas_cluster_map.crs is not None and str(areas_cluster_map.crs) != "EPSG:4326":
    areas_cluster_map = areas_cluster_map.to_crs(epsg=4326)

# Centro del mapa
centro = areas_cluster_map.geometry.unary_union.centroid
lat_centro = centro.y
lon_centro = centro.x

# Crear mapa base
mapa = folium.Map(
    location=[lat_centro, lon_centro],
    zoom_start=5.5,
    tiles="cartodbpositron"
)

# Función de estilo
def estilo(feature):
    return {
        "fillColor": feature["properties"].get("color", "#808080"),
        "color": "#444444",
        "weight": 0.4,
        "fillOpacity": 0.75
    }

# Tooltip interactivo
tooltip = folium.GeoJsonTooltip(
    fields=["ID_GRUPO", "cluster"],
    aliases=["ID_GRUPO:", "Cluster:"],
    localize=True,
    sticky=False,
    labels=True
)

# Añadir capa geográfica
folium.GeoJson(
    areas_cluster_map[["ID_GRUPO", "cluster", "color", "geometry"]].to_json(),
    name="Comunidades",
    style_function=estilo,
    tooltip=tooltip
).add_to(mapa)

folium.LayerControl().add_to(mapa)

mapa